<a href="https://colab.research.google.com/github/tomonari-masada/course2026-sml/blob/main/10_document_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 文書分類

* IMDBデータセットを使った感情分析
  * 映画レビューを、映画に肯定的か否定的かで、2値に分類する。

* 本演習の目的
  1. 検証データでできるかぎりチューニングを行い、最後にテストデータでの分類性能を明らかにする。
  2. 肯定的なレビューと否定的なレビューとを分類する際に、どのような単語が特に効いているか明らかにする。


## 準備


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    RocCurveDisplay, PrecisionRecallDisplay,
    ConfusionMatrixDisplay, classification_report
)
from sklearn.feature_extraction.text import TfidfVectorizer
from datasets import load_dataset

%config InlineBackend.figure_format = 'retina'

import warnings
warnings.filterwarnings('ignore')

### IMDBデータセットの取得

* `datasets`ライブラリを使ってIMDBデータセットを取得する。
  * 訓練データ：25,000件、テストデータ：25,000件。

In [ ]:
dataset = load_dataset("stanfordnlp/imdb")

In [ ]:
dataset["train"]

In [ ]:
# テキストとラベルの先頭5件を確認
dataset["train"]["text"][:5]

In [ ]:
dataset["train"]["label"][:5]

* 後で扱いやすいようにNumPyの配列に変換しておく。


In [ ]:
train_corpus = np.array(dataset["train"]["text"])
y_train      = np.array(dataset["train"]["label"])

test_corpus  = np.array(dataset["test"]["text"])
y_test       = np.array(dataset["test"]["label"])

print(f"訓練データ: {len(train_corpus)} 件  （正例: {y_train.sum()}, 負例: {(1-y_train).sum()}）")
print(f"テストデータ: {len(test_corpus)} 件  （正例: {y_test.sum()}, 負例: {(1-y_test).sum()}）")

## TF-IDFによるテキストの埋め込み

### TF-IDFとは

TF-IDFはテキストをベクトル化する古典的な手法で、ベクトルの次元は語彙数となる。

各単語 $t$、各文書 $d$、全文書数 $N$、単語 $t$ が出現する文書数 $\mathrm{df}(t)$ として、scikit-learnでは以下の式で計算される：

$$\mathrm{tf\text{-}idf}(t, d) = \mathrm{tf}(t, d) \cdot \left(\log\frac{1 + N}{1 + \mathrm{df}(t)} + 1\right)$$

* **TF**（Term Frequency）: 文書 $d$ 中での単語 $t$ の出現回数。
* **IDF**（Inverse Document Frequency）: 多数の文書に現れる単語は重みを下げる。scikit-learnでは分母・分子に1を加えてゼロ除算を回避し、さらに1を加えて非負を保証する。
* 特定の文書に頻出する単語ほど TF-IDF が大きくなるが、全体的にどこでも出現するありふれた単語は IDF によって抑制される。


### `TfidfVectorizer` の使い方と注意点

* データリークに注意
  * `TfidfVectorizer` は `fit` 時に語彙と IDF を「学習」する。
  * したがって、**訓練データのみに `fit` し、検証・テストデータには `transform` だけを適用する**こと。
  * 検証・テストデータでも `fit` してしまうと、本来知ってはいけない情報（テストデータ中の単語頻度）がモデルに漏れ込む（データリーク）。
  * 後述する `Pipeline` を使うと、この処理を安全かつ簡潔に記述できる。

* 主なパラメータ

| パラメータ | 説明 |
|---|---|
| `stop_words='english'` | the, a, is などの英語のストップワードを語彙から除外する |
| `min_df` | 指定割合より少ない文書にしか出現しない単語を除外（希少語の削減） |
| `max_df` | 指定割合より多い文書に出現する単語を除外（ありふれた語の削減） |
| `max_features` | 語彙の上限数 |

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english')
X_train_sample = vectorizer.fit_transform(dataset["train"]["text"])
print('X_train: 文書数 {}, 語彙数 {}'.format(*X_train_sample.shape))

In [ ]:
# 語彙を取得する（アルファベット順）
vocab = np.array(vectorizer.get_feature_names_out())
print(vocab[2000:2020])

In [ ]:
# 最初の文書のTF-IDF表現（スパース行列）
print(type(X_train_sample))
print(X_train_sample[0])

## Pipeline を使った実装

`sklearn.pipeline.Pipeline` を使うと、TF-IDFベクトル化と分類器を一つのオブジェクトにまとめられる。

**メリット：**
1. **データリークの防止**：交差検証の各foldで `fit` の範囲が自動的に訓練データのみに限定される。
2. **コードの簡潔化**：変換・学習・予測が1行で書ける。
3. **ハイパーパラメータ探索との統合**：`GridSearchCV` と組み合わせやすい。

以降の交差検証では Pipeline を使って実装する。


## ロジスティック回帰のチューニング

### L2正則化

* 交差検証で正則化パラメータ `C` をチューニングする。


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1234)

print("=" * 55)
print(f"{'C':>10}  {'mean acc':>10}  {'std':>8}")
print("=" * 55)

for C in 10. ** np.arange(-1, 4):
    scores = []
    for train_index, valid_index in skf.split(train_corpus, y_train):
        # Pipeline により、各foldの fit は訓練データのみに適用される（データリーク防止）
        pipe = Pipeline([
            ('tfidf', TfidfVectorizer(stop_words='english')),
            ('clf',   LogisticRegression(C=C, solver='liblinear', random_state=123))
        ])
        pipe.fit(train_corpus[train_index], y_train[train_index])
        score = pipe.score(train_corpus[valid_index], y_train[valid_index])
        scores.append(score)
    scores = np.array(scores)
    print(f"  C={C:.2e}  mean={scores.mean():.4f}  std={scores.std():.4f}")

### L1正則化

* L1正則化は係数をスパースにする（不要な特徴量の係数をちょうど0にする）。
  * L2正則化と比較して、どちらが良いか確認してみよう。


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1234)

print("=" * 55)
print(f"{'C':>10}  {'mean acc':>10}  {'std':>8}")
print("=" * 55)

for C in 10. ** np.arange(-1, 4):
    scores = []
    for train_index, valid_index in skf.split(train_corpus, y_train):
        pipe = Pipeline([
            ('tfidf', TfidfVectorizer(stop_words='english')),
            ('clf',   LogisticRegression(penalty='l1', C=C, solver='liblinear', random_state=123))
        ])
        pipe.fit(train_corpus[train_index], y_train[train_index])
        score = pipe.score(train_corpus[valid_index], y_train[valid_index])
        scores.append(score)
    scores = np.array(scores)
    print(f"  C={C:.2e}  mean={scores.mean():.4f}  std={scores.std():.4f}")

## TF-IDFのチューニング

* `min_df` を変えて語彙のフィルタリングが精度に与える影響を確認する。
  * `max_df` も合わせて変えてみよう。


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1234)

print("=" * 60)
print(f"{'min_df':>12}  {'mean acc':>10}  {'std':>8}")
print("=" * 60)

for min_df in [0.0, 1/5000, 1/2000, 1/1000]:
    scores = []
    for train_index, valid_index in skf.split(train_corpus, y_train):
        pipe = Pipeline([
            ('tfidf', TfidfVectorizer(stop_words='english', min_df=min_df)),
            ('clf',   LogisticRegression(C=1.0, solver='liblinear', random_state=123))
        ])
        pipe.fit(train_corpus[train_index], y_train[train_index])
        score = pipe.score(train_corpus[valid_index], y_train[valid_index])
        scores.append(score)
    scores = np.array(scores)
    print(f"  min_df={min_df:.5f}  mean={scores.mean():.4f}  std={scores.std():.4f}")

## テストセットで評価

* チューニングで見つけたベストの設定を以下の変数に記入してから実行すること。

* テストデータは最後の一回だけ使う！
  * チューニングはすべて訓練データ内の交差検証で行い、テストデータは最終評価の一回のみ使用すること。
  * テスト結果を見てからパラメータを調整するのは「テスト汚染」にあたる。


In [ ]:
# ──────────────────────────────────────────────
# ★ チューニングで見つけたベスト設定をここに記入 ★
BEST_C      = 1.0          # 正則化パラメータ
BEST_PENALTY = 'l2'        # 'l1' or 'l2'
BEST_MIN_DF  = 0.0         # TfidfVectorizer の min_df
# ──────────────────────────────────────────────

best_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', min_df=BEST_MIN_DF)),
    ('clf',   LogisticRegression(penalty=BEST_PENALTY, C=BEST_C,
                                 solver='liblinear', random_state=123))
])
best_pipe.fit(train_corpus, y_train)

test_acc = best_pipe.score(test_corpus, y_test)
print(f"test accuracy: {test_acc:.4f}")

### 詳細な分類レポート

* `classification_report` で適合率・再現率・F1スコアを確認する。

  * **適合率 (Precision)**：「陽性と予測したもののうち本当に陽性の割合」
  * **再現率 (Recall)**：「本当の陽性のうちモデルが陽性と予測できた割合」
  * **F1スコア**：適合率と再現率の調和平均


In [ ]:
y_pred = best_pipe.predict(test_corpus)
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

### 混同行列

* 混同行列（Confusion Matrix）で誤分類のパターンを確認する。

  * **True Positive (TP)**：正例を正例と正しく予測
  * **True Negative (TN)**：負例を負例と正しく予測
  * **False Positive (FP)**：負例を正例と誤予測（偽陽性）
  * **False Negative (FN)**：正例を負例と誤予測（偽陰性）

* どの種類の誤りが多いか確認しよう。


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Negative', 'Positive'],
    ax=ax
)
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()


### ROCカーブ と PR（適合率-再現率）カーブ

* **ROCカーブ**：FP率（偽陽性率）と TP率（真陽性率＝再現率）のトレードオフ。
  * AUCが1に近いほど良い。クラス不均衡が小さい場合に有効。
* **PRカーブ**：再現率と適合率のトレードオフ。
  * クラス不均衡が大きい場合に特に有効（陽性クラスが少ない場合、ROCは楽観的に見えることがある）。

* IMDBはほぼ均等クラスなので両者の差は小さいが、どちらがどんな状況で有用かを意識しよう。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

RocCurveDisplay.from_estimator(
    best_pipe, test_corpus, y_test,
    name="Logistic Regression", ax=axes[0]
)
axes[0].set_title("ROC Curve")

PrecisionRecallDisplay.from_estimator(
    best_pipe, test_corpus, y_test,
    name="Logistic Regression", ax=axes[1]
)
axes[1].set_title("Precision-Recall Curve")

plt.tight_layout()
plt.show()


## SVM（サポートベクターマシン）

ロジスティック回帰と同様に、`LinearSVC` でチューニング・評価・解釈を行う。

**ヒント：**
1. `LinearSVC` の正則化パラメータは `C`（ロジスティック回帰と同じ意味）。
2. `LinearSVC` には `predict_proba` がないため `RocCurveDisplay.from_estimator` は使えない。代わりに `decision_function` を使う方法を調べてみよう。
3. 係数は `clf.coef_` で取得できる（ロジスティック回帰と同じ）。


In [ ]:
# === SVM のチューニング（ここを実装してみよう） ===

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1234)

# TODO: 下の ??? をうめて、C をグリッドサーチせよ
# for C in ???:
#     pipe = Pipeline([
#         ('tfidf', TfidfVectorizer(stop_words='english')),
#         ('clf',   LinearSVC(C=C, random_state=123, max_iter=2000))
#     ])
#     ...


In [ ]:
# === SVM のテストセット評価（ここを実装してみよう） ===

# BEST_C_SVM = ???   # チューニングで見つけたベスト値

# svm_pipe = Pipeline([...])
# svm_pipe.fit(train_corpus, y_train)
# print(classification_report(y_test, svm_pipe.predict(test_corpus),
#                             target_names=['Negative', 'Positive']))


## ロジスティック回帰による分類に効いている単語を調べる

* 係数の絶対値が大きい単語は分類に強く寄与している。
  * 正の係数 → 肯定的レビューへの寄与
  * 負の係数 → 否定的レビューへの寄与

### 方法1：ロジスティック回帰の係数をそのまま可視化する


In [ ]:
# テスト評価用のパイプラインで語彙・係数を取得
vectorizer_lr = best_pipe.named_steps['tfidf']
clf_lr        = best_pipe.named_steps['clf']
vocab_lr      = np.array(vectorizer_lr.get_feature_names_out())
coef          = clf_lr.coef_[0]

n_features = 30
positions  = np.arange(n_features)

# 正・負の係数上位30語を一枚のグラフにまとめる
top_pos_idx = np.argsort(-coef)[:n_features]
top_neg_idx = np.argsort( coef)[:n_features]

fig, axes = plt.subplots(1, 2, figsize=(16, 9))

# 肯定的レビューに効く単語
axes[0].barh(positions, coef[top_pos_idx][::-1], align='center', color='steelblue')
axes[0].set_yticks(positions)
axes[0].set_yticklabels(vocab_lr[top_pos_idx][::-1])
axes[0].set_xlabel('Coefficient')
axes[0].set_title('Top words for Positive reviews')

# 否定的レビューに効く単語
axes[1].barh(positions, coef[top_neg_idx], align='center', color='salmon')
axes[1].set_yticks(positions)
axes[1].set_yticklabels(vocab_lr[top_neg_idx])
axes[1].set_xlabel('Coefficient')
axes[1].set_title('Top words for Negative reviews')

plt.tight_layout()
plt.show()

### 方法2：正負の係数を一つのグラフに表示する

* 正負合わせて上位単語を一枚のグラフにまとめると、両方向のインパクトを直感的に比較できる。


In [ ]:
n_each = 15  # 正・負それぞれ上位15語

top_pos_idx = np.argsort(-coef)[:n_each]
top_neg_idx = np.argsort( coef)[:n_each]
combined_idx = np.concatenate([top_neg_idx[::-1], top_pos_idx])

words  = vocab_lr[combined_idx]
values = coef[combined_idx]
colors = ['salmon' if v < 0 else 'steelblue' for v in values]

plt.figure(figsize=(10, 10))
plt.barh(np.arange(len(words)), values, color=colors, align='center')
plt.yticks(np.arange(len(words)), words)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Coefficient')
plt.title('Logistic Regression: Top Positive & Negative Coefficients')
plt.tight_layout()
plt.show()

## SVMによる分類に効いている単語を調べる

* ロジスティック回帰の可視化コードを参考に、SVMでも実装してみよう。
  * `LinearSVC` の係数は `svm_pipe.named_steps['clf'].coef_[0]` で取得できる。


In [ ]:
# === SVM の係数可視化（ここを実装してみよう） ===

# vocab_svm = np.array(svm_pipe.named_steps['tfidf'].get_feature_names_out())
# coef_svm  = svm_pipe.named_steps['clf'].coef_[0]
# ...


## 課題

* SVMによる分類（上で空欄にしてある部分）と分類器の解釈を実践してみよう。